# Config A: ViSoBERT Baseline Reproduction

Tai lap paper baseline: ViSoBERT + Scenario 1 + BCE + pos_weight + threshold 0.5

Target: Macro F1 ~61.50% (sai so <= 0.5%)

In [ ]:
import os, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
import ast

## Hyperparameters (locked -- do not change between runs)

In [ ]:
CFG = {
    "model_name"   : "uitnlp/visobert",
    "num_labels"   : 28,
    "max_len"      : 128,
    "batch_size"   : 32,
    "epochs"       : 12,
    "lr"           : 5e-5,
    "dropout"      : 0.2,
    "threshold"    : 0.5,
    "seed"         : 42,
    "scenario"     : 1,
    "run_name"     : "A_seed42_scenario1",
}

LABEL_NAMES = [
    "amusement","excitement","joy","love","desire","optimism",
    "caring","pride","admiration","gratitude","relief","approval",
    "realization","surprise","curiosity","confusion","fear",
    "nervousness","remorse","embarrassment","disappointment",
    "sadness","grief","disgust","anger","annoyance","disapproval","neutral"
]

OPERATIONAL = [20, 24, 16, 22]  # disappointment, anger, fear, grief

print(json.dumps(CFG, indent=2))

## Seed

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Seed: {CFG['seed']}")

## Load Data

In [ ]:
DATA_DIR = Path("../data/raw")

train_df = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")
val_df   = pd.read_csv(DATA_DIR / "val.csv",   encoding="utf-8-sig")
test_df  = pd.read_csv(DATA_DIR / "test.csv",  encoding="utf-8-sig")

for df in [train_df, val_df, test_df]:
    df["labels"] = df["labels"].apply(ast.literal_eval)

print(f"train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

## pos_weight

Cong thuc tu paper: `pos_weight_i = (N - n_i) / n_i`

In [ ]:
N = len(train_df)
label_counts = np.zeros(CFG["num_labels"], dtype=np.float32)
for labels in train_df["labels"]:
    for l in labels:
        label_counts[l] += 1

pos_weight = (N - label_counts) / label_counts
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32).to(DEVICE)

print("pos_weight range:", pos_weight.min().round(2), "to", pos_weight.max().round(2))
for name, w in zip(LABEL_NAMES, pos_weight):
    print(f"  {name:20s}: {w:.2f}")

## Dataset & DataLoader

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, num_labels):
        self.texts     = df["text"].tolist()
        self.labels    = df["labels"].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.num_labels = num_labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        label_vec = torch.zeros(self.num_labels, dtype=torch.float32)
        for l in self.labels[idx]:
            label_vec[l] = 1.0
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         label_vec,
        }

train_ds = EmotionDataset(train_df, tokenizer, CFG["max_len"], CFG["num_labels"])
val_ds   = EmotionDataset(val_df,   tokenizer, CFG["max_len"], CFG["num_labels"])
test_ds  = EmotionDataset(test_df,  tokenizer, CFG["max_len"], CFG["num_labels"])

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"], shuffle=False, num_workers=0, pin_memory=True)

print(f"Batches -- train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")

## Model

In [ ]:
class ViSoBERTClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.2):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        out    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0, :]  # CLS token
        pooled = self.dropout(pooled)
        return self.classifier(pooled)            # raw logits

model = ViSoBERTClassifier(CFG["model_name"], CFG["num_labels"], CFG["dropout"]).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Optimizer, Scheduler, Loss

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"])

total_steps  = CFG["epochs"] * len(train_loader)
warmup_steps = len(train_loader)   # 1 epoch warmup (matches paper)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

print(f"Total steps: {total_steps}, warmup: {warmup_steps}")

## Eval Function

In [ ]:
def evaluate(loader, model, threshold=0.5):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            logits = model(input_ids, attention_mask)
            probs  = torch.sigmoid(logits).cpu().numpy()
            preds  = (probs >= threshold).astype(int)
            all_preds.append(preds)
            all_labels.append(batch["labels"].numpy())
    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_labels)
    macro = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    wtd   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    return macro, wtd, y_pred, y_true

## Training Loop

In [ ]:
RUN_DIR = Path(f"../runs/{CFG['run_name']}")
RUN_DIR.mkdir(parents=True, exist_ok=True)

best_macro = 0.0
train_log  = []

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    val_macro, val_wtd, _, _ = evaluate(val_loader, model, CFG["threshold"])

    print(f"Epoch {epoch:2d} | loss {avg_loss:.4f} | val Macro F1 {val_macro:.4f} | val Weighted {val_wtd:.4f}")
    train_log.append({"epoch": epoch, "loss": avg_loss, "val_macro": val_macro, "val_weighted": val_wtd})

    if val_macro > best_macro:
        best_macro = val_macro
        torch.save(model.state_dict(), RUN_DIR / "best_model.pt")
        print(f"  --> Saved best (val Macro F1 = {best_macro:.4f})")

pd.DataFrame(train_log).to_csv(RUN_DIR / "train_log.csv", index=False)
print(f"\nBest val Macro F1: {best_macro:.4f}")

## Test Evaluation

In [ ]:
model.load_state_dict(torch.load(RUN_DIR / "best_model.pt", weights_only=True))

test_macro, test_wtd, y_pred_test, y_true_test = evaluate(test_loader, model, CFG["threshold"])
val_macro_f, val_wtd_f, y_pred_val, y_true_val = evaluate(val_loader,  model, CFG["threshold"])

print("=" * 50)
print("TEST SET (best checkpoint)")
print(f"  Macro F1    = {test_macro:.4f}")
print(f"  Weighted F1 = {test_wtd:.4f}")
print("=" * 50)
print(f"Paper target  : Macro F1 = 0.6150")
print(f"Gap           : {test_macro - 0.6150:+.4f}")

## Save Artifacts

In [ ]:
np.save(RUN_DIR / "val_predictions.npy",  y_pred_val)
np.save(RUN_DIR / "test_predictions.npy", y_pred_test)
np.save(RUN_DIR / "val_true.npy",  y_true_val)
np.save(RUN_DIR / "test_true.npy", y_true_test)

per_label_f1 = f1_score(y_true_test, y_pred_test, average=None, zero_division=0)
per_label_df = pd.DataFrame({"label": LABEL_NAMES, "f1": per_label_f1.round(4)})
per_label_df.to_csv(RUN_DIR / "per_label_f1.csv", index=False)

op_recall = {}
for idx in OPERATIONAL:
    mask = y_true_test[:, idx] == 1
    if mask.sum() > 0:
        op_recall[LABEL_NAMES[idx]] = float(y_pred_test[mask, idx].mean().round(4))

metrics = {
    "config": "A",
    "seed": CFG["seed"],
    "scenario": CFG["scenario"],
    "test_macro_f1": round(test_macro, 4),
    "test_weighted_f1": round(test_wtd, 4),
    "val_macro_f1": round(val_macro_f, 4),
    "operational_recall": op_recall,
}
with open(RUN_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
with open(RUN_DIR / "config.json", "w") as f:
    json.dump(CFG, f, indent=2)

print("Saved to:", RUN_DIR)
print(json.dumps(metrics, indent=2))